In [1]:
# ==================================================
# LOAD DATA FOR PROPERTY RISK
# ==================================================

dim_property = spark.read.parquet(
    minio_path("gold/data_model/dim_property")
)

fact_311 = spark.read.parquet(
    minio_path("gold/data_model/fact_311_event")
)

fact_hpd = spark.read.parquet(
    minio_path("gold/data_model/fact_hpd_violation")
)

building_risk = spark.read.parquet(
    minio_path("gold/building_risk/building_risk_score")
)


print("dim_property:", dim_property.count())
print("fact_311:", fact_311.count())
print("fact_hpd:", fact_hpd.count())
print("building_risk:", building_risk.count())

NameError: name 'spark' is not defined

In [2]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)


# ==================================================
# IMPORT PROJECT HELPERS
# ==================================================

from minio_config import configure_minio, minio_path


# ==================================================
# CREATE SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Property Risk")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)


# ==================================================
# CONFIGURE MINIO
# ==================================================

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")


# ==================================================
# TEST
# ==================================================

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/12 14:24:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/12 14:24:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/12 14:24:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 3.4.0
Master: local[2]
Test: 1


In [3]:
# ==================================================
# LOAD DATA FOR PROPERTY RISK
# ==================================================

dim_property = spark.read.parquet(
    minio_path("gold/data_model/dim_property")
)

fact_311 = spark.read.parquet(
    minio_path("gold/data_model/fact_311_event")
)

fact_hpd = spark.read.parquet(
    minio_path("gold/data_model/fact_hpd_violation")
)

building_risk = spark.read.parquet(
    minio_path("gold/building_risk/building_risk_score")
)

print("dim_property:", dim_property.count())
print("fact_311:", fact_311.count())
print("fact_hpd:", fact_hpd.count())
print("building_risk:", building_risk.count())

26/09/12 14:24:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


dim_property: 858284
fact_311: 885306
fact_hpd: 927308
building_risk: 197958


In [4]:
# ==================================================
# PROPERTY-ONLY EVENTS
# property_id exists, but building_id is unresolved
# ==================================================

property_only_311 = (
    fact_311
    .filter(
        F.col("property_id").isNotNull()
        & F.col("building_id").isNull()
    )
)

property_only_hpd = (
    fact_hpd
    .filter(
        F.col("property_id").isNotNull()
        & F.col("building_id").isNull()
    )
)


# ==================================================
# VALIDATION
# ==================================================

print(
    "311 property-only events:",
    property_only_311.count()
)

print(
    "311 properties affected:",
    property_only_311
    .select("property_id")
    .distinct()
    .count()
)

print(
    "HPD property-only events:",
    property_only_hpd.count()
)

print(
    "HPD properties affected:",
    property_only_hpd
    .select("property_id")
    .distinct()
    .count()
)

311 property-only events: 39652


311 properties affected: 13858


HPD property-only events: 796


HPD properties affected: 146


In [5]:
from datetime import date

# ==================================================
# PROPERTY RISK AS-OF DATE
# Keep identical to Building Risk
# ==================================================

risk_as_of_date = date(2026, 8, 19)

print("Property Risk as-of date:", risk_as_of_date)

Property Risk as-of date: 2026-08-19


In [6]:
# ==================================================
# PREPARE PROPERTY-ONLY 311 EVENTS
# ==================================================

property_311_risk = (
    property_only_311

    .withColumn(
        "event_age_days",
        F.datediff(
            F.lit(str(risk_as_of_date)),
            F.to_date("created_date")
        )
    )

    .filter(
        F.col("event_age_days") >= 0
    )
)


# ==================================================
# 311 FEATURES PER PROPERTY
# ==================================================

property_features_311 = (
    property_311_risk

    .groupBy("property_id")

    .agg(
        F.sum(
            F.when(
                F.col("event_age_days").between(0, 30),
                1
            ).otherwise(0)
        ).alias("property_311_0_30"),

        F.sum(
            F.when(
                F.col("event_age_days").between(31, 90),
                1
            ).otherwise(0)
        ).alias("property_311_31_90"),

        F.sum(
            F.when(
                F.col("event_age_days").between(91, 365),
                1
            ).otherwise(0)
        ).alias("property_311_91_365")
    )
)

In [7]:
# ==================================================
# PREPARE PROPERTY-ONLY HPD EVENTS
# ==================================================

property_hpd_risk = (
    property_only_hpd

    .withColumn(
        "event_age_days",
        F.datediff(
            F.lit(str(risk_as_of_date)),
            F.to_date("inspectiondate")
        )
    )

    .filter(
        F.col("event_age_days") >= 0
    )
)


# ==================================================
# HPD FEATURES PER PROPERTY
# ==================================================

property_features_hpd = (
    property_hpd_risk

    .groupBy("property_id")

    .agg(
        # Active violations by severity
        F.sum(
            F.when(
                (F.col("violationstatus") == "OPEN")
                & (F.col("class") == "A"),
                1
            ).otherwise(0)
        ).alias("property_hpd_open_class_a"),

        F.sum(
            F.when(
                (F.col("violationstatus") == "OPEN")
                & (F.col("class") == "B"),
                1
            ).otherwise(0)
        ).alias("property_hpd_open_class_b"),

        F.sum(
            F.when(
                (F.col("violationstatus") == "OPEN")
                & (F.col("class") == "C"),
                1
            ).otherwise(0)
        ).alias("property_hpd_open_class_c"),

        # Recency buckets
        F.sum(
            F.when(
                F.col("event_age_days").between(0, 30),
                1
            ).otherwise(0)
        ).alias("property_hpd_0_30"),

        F.sum(
            F.when(
                F.col("event_age_days").between(31, 90),
                1
            ).otherwise(0)
        ).alias("property_hpd_31_90"),

        F.sum(
            F.when(
                F.col("event_age_days").between(91, 365),
                1
            ).otherwise(0)
        ).alias("property_hpd_91_365")
    )
)

In [8]:
print(
    "Properties with 311-only features:",
    property_features_311.count()
)

print(
    "Properties with HPD-only features:",
    property_features_hpd.count()
)

property_features_311.orderBy(
    F.desc("property_311_0_30")
).show(
    10,
    truncate=False
)

property_features_hpd.orderBy(
    F.desc("property_hpd_open_class_c")
).show(
    10,
    truncate=False
)

Properties with 311-only features: 13725
Properties with HPD-only features: 146


+---------------+-----------------+------------------+-------------------+
|property_id    |property_311_0_30|property_311_31_90|property_311_91_365|
+---------------+-----------------+------------------+-------------------+
|PROP:2051000017|86               |97                |16                 |
|PROP:2031560072|26               |50                |47                 |
|PROP:3048680055|20               |0                 |0                  |
|PROP:1009720001|18               |13                |87                 |
|PROP:3051710027|17               |0                 |6                  |
|PROP:3080540051|17               |0                 |0                  |
|PROP:1010780056|15               |0                 |2                  |
|PROP:5062520062|15               |0                 |0                  |
|PROP:2044760036|15               |0                 |0                  |
|PROP:2051410120|14               |5                 |43                 |
+---------------+--------

In [9]:
# ==================================================
# AGGREGATE BUILDING RISK TO PROPERTY LEVEL
# ==================================================

property_building_risk = (
    building_risk

    .filter(
        F.col("property_id").isNotNull()
    )

    .groupBy("property_id")

    .agg(
        F.countDistinct(
            "building_id"
        ).alias("building_count"),

        F.max(
            "building_risk_score"
        ).alias("max_building_risk_score"),

        F.avg(
            "building_risk_score"
        ).alias("avg_building_risk_score")
    )
)

In [10]:
# ==================================================
# PROPERTY-ONLY RAW RISK SIGNALS
# ==================================================

property_311_signals = (
    property_features_311

    .withColumn(
        "property_risk_311_raw",
        F.col("property_311_0_30") * 3
        + F.col("property_311_31_90") * 2
        + F.col("property_311_91_365")
    )
)


property_hpd_signals = (
    property_features_hpd

    .withColumn(
        "property_risk_hpd_severity_raw",

        F.col("property_hpd_open_class_a")
        + F.col("property_hpd_open_class_b") * 2
        + F.col("property_hpd_open_class_c") * 4
    )

    .withColumn(
        "property_risk_hpd_recency_raw",

        F.col("property_hpd_0_30") * 3
        + F.col("property_hpd_31_90") * 2
        + F.col("property_hpd_91_365")
    )
)

In [11]:
# ==================================================
# BUILD PROPERTY RISK FEATURES
# Grain: 1 row = 1 property_id
# ==================================================

property_risk_features = (
    dim_property

    .join(
        property_building_risk,
        on="property_id",
        how="left"
    )

    .join(
        property_311_signals,
        on="property_id",
        how="left"
    )

    .join(
        property_hpd_signals,
        on="property_id",
        how="left"
    )
)

In [12]:
property_event_columns = [
    "property_311_0_30",
    "property_311_31_90",
    "property_311_91_365",
    "property_risk_311_raw",

    "property_hpd_open_class_a",
    "property_hpd_open_class_b",
    "property_hpd_open_class_c",

    "property_hpd_0_30",
    "property_hpd_31_90",
    "property_hpd_91_365",

    "property_risk_hpd_severity_raw",
    "property_risk_hpd_recency_raw"
]

property_risk_features = (
    property_risk_features
    .fillna(
        0,
        subset=property_event_columns
    )
)

In [13]:
print(
    "Property risk feature rows:",
    property_risk_features.count()
)

print(
    "Distinct property_id:",
    property_risk_features
    .select("property_id")
    .distinct()
    .count()
)

print(
    "Properties with Building Risk:",
    property_risk_features
    .filter(
        F.col("max_building_risk_score").isNotNull()
    )
    .count()
)

print(
    "Properties with property-only 311 Risk:",
    property_risk_features
    .filter(
        F.col("property_risk_311_raw") > 0
    )
    .count()
)

print(
    "Properties with property-only HPD Risk:",
    property_risk_features
    .filter(
        (F.col("property_risk_hpd_severity_raw") > 0)
        | (F.col("property_risk_hpd_recency_raw") > 0)
    )
    .count()
)

Property risk feature rows: 858284
Distinct property_id: 858284
Properties with Building Risk: 171582


Properties with property-only 311 Risk: 13725


Properties with property-only HPD Risk: 146


In [14]:
# ==================================================
# PROPERTY-ONLY RISK DISTRIBUTION
# ==================================================

property_risk_columns = [
    "property_risk_311_raw",
    "property_risk_hpd_severity_raw",
    "property_risk_hpd_recency_raw"
]

percentiles = [
    0.50,
    0.90,
    0.95,
    0.975,
    0.99,
    0.995
]

for column_name in property_risk_columns:

    quantiles = (
        property_risk_features
        .filter(F.col(column_name) > 0)
        .approxQuantile(
            column_name,
            percentiles,
            0.001
        )
    )

    print(
        "\n",
        column_name,
        "\nP50   =", quantiles[0],
        "\nP90   =", quantiles[1],
        "\nP95   =", quantiles[2],
        "\nP97.5 =", quantiles[3],
        "\nP99   =", quantiles[4],
        "\nP99.5 =", quantiles[5]
    )


 property_risk_311_raw 
P50   = 2.0 
P90   = 8.0 
P95   = 12.0 
P97.5 = 16.0 
P99   = 24.0 
P99.5 = 31.0

 property_risk_hpd_severity_raw 
P50   = 3.0 
P90   = 31.0 
P95   = 38.0 
P97.5 = 68.0 
P99   = 170.0 
P99.5 = 170.0

 property_risk_hpd_recency_raw 
P50   = 1.0 
P90   = 17.0 
P95   = 26.0 
P97.5 = 43.0 
P99   = 110.0 
P99.5 = 115.0


In [15]:
# ==================================================
# PROPERTY RISK COVERAGE PROFILE
# ==================================================

has_property_only_risk = (
    (F.col("property_risk_311_raw") > 0)
    |
    (F.col("property_risk_hpd_severity_raw") > 0)
    |
    (F.col("property_risk_hpd_recency_raw") > 0)
)


print(
    "Building Risk + Property-only Risk:",
    property_risk_features
    .filter(
        F.col("max_building_risk_score").isNotNull()
        & has_property_only_risk
    )
    .count()
)

print(
    "Property-only Risk WITHOUT Building Risk:",
    property_risk_features
    .filter(
        F.col("max_building_risk_score").isNull()
        & has_property_only_risk
    )
    .count()
)

Building Risk + Property-only Risk: 532


Property-only Risk WITHOUT Building Risk: 13333


In [16]:
# ==================================================
# PROPERTY-ONLY ROBUST NORMALIZATION
# ==================================================

CAP_PROPERTY_311 = 24.0
CAP_PROPERTY_HPD_SEVERITY = 170.0
CAP_PROPERTY_HPD_RECENCY = 110.0


def log_normalize(column_name, cap_value):
    return (
        F.log1p(
            F.least(
                F.col(column_name),
                F.lit(cap_value)
            )
        )
        / F.log1p(F.lit(cap_value))
        * 100
    )


property_risk_normalized = (
    property_risk_features

    .withColumn(
        "property_score_311",
        log_normalize(
            "property_risk_311_raw",
            CAP_PROPERTY_311
        )
    )

    .withColumn(
        "property_score_hpd_severity",
        log_normalize(
            "property_risk_hpd_severity_raw",
            CAP_PROPERTY_HPD_SEVERITY
        )
    )

    .withColumn(
        "property_score_hpd_recency",
        log_normalize(
            "property_risk_hpd_recency_raw",
            CAP_PROPERTY_HPD_RECENCY
        )
    )
)

In [17]:
# ==================================================
# PROPERTY-ONLY RISK SCORE
# Availability-aware weighting
# ==================================================

property_risk_normalized = (
    property_risk_normalized

    .withColumn(
        "has_property_311",
        F.when(
            F.col("property_risk_311_raw") > 0,
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "has_property_hpd",
        F.when(
            (F.col("property_risk_hpd_severity_raw") > 0)
            | (F.col("property_risk_hpd_recency_raw") > 0),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "property_risk_weight_available",
        F.col("has_property_311") * 25
        + F.col("has_property_hpd") * 40
    )

    .withColumn(
        "property_only_risk_score",
        F.when(
            F.col("property_risk_weight_available") > 0,

            (
                F.col("property_score_311")
                * F.col("has_property_311") * 25

                + F.col("property_score_hpd_severity")
                * F.col("has_property_hpd") * 28

                + F.col("property_score_hpd_recency")
                * F.col("has_property_hpd") * 12
            )
            /
            F.col("property_risk_weight_available")
        )
    )
)

In [18]:
property_only_quantiles = (
    property_risk_normalized
    .filter(
        F.col("property_only_risk_score").isNotNull()
    )
    .approxQuantile(
        "property_only_risk_score",
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ],
        0.001
    )
)

print("Property-only Risk Distribution")
print("P50 =", property_only_quantiles[0])
print("P75 =", property_only_quantiles[1])
print("P90 =", property_only_quantiles[2])
print("P95 =", property_only_quantiles[3])
print("P99 =", property_only_quantiles[4])

Property-only Risk Distribution
P50 = 34.13030972429926
P75 = 50.0
P90 = 68.26061944859853
P95 = 79.68463205835413
P99 = 100.0


In [19]:
property_risk_normalized.select(
    "property_id",
    "max_building_risk_score",
    "property_risk_311_raw",
    "property_risk_hpd_severity_raw",
    "property_risk_hpd_recency_raw",
    "property_only_risk_score"
).filter(
    F.col("property_only_risk_score").isNotNull()
).orderBy(
    F.desc("property_only_risk_score")
).show(
    20,
    truncate=False
)

+---------------+-----------------------+---------------------+------------------------------+-----------------------------+------------------------+
|property_id    |max_building_risk_score|property_risk_311_raw|property_risk_hpd_severity_raw|property_risk_hpd_recency_raw|property_only_risk_score|
+---------------+-----------------------+---------------------+------------------------------+-----------------------------+------------------------+
|PROP:2023260025|null                   |27                   |0                             |0                            |100.0                   |
|PROP:2029670006|31.06                  |0                    |170                           |110                          |100.0                   |
|PROP:2057670798|null                   |24                   |0                             |0                            |100.0                   |
|PROP:3007500043|null                   |24                   |0                             |0     

In [20]:
# ==================================================
# VALIDATE SAVED BUILDING RISK
# ==================================================

building_risk.select(
    F.min("building_risk_score").alias("min_score"),
    F.max("building_risk_score").alias("max_score"),
    F.avg("building_risk_score").alias("avg_score")
).show()

print(
    "Buildings with score > 100:",
    building_risk
    .filter(F.col("building_risk_score") > 100)
    .count()
)

print(
    "Buildings with score < 0:",
    building_risk
    .filter(F.col("building_risk_score") < 0)
    .count()
)

+---------+---------+------------------+
|min_score|max_score|         avg_score|
+---------+---------+------------------+
|     0.69|    97.88|19.842354590367236|
+---------+---------+------------------+

Buildings with score > 100: 0
Buildings with score < 0: 0


In [21]:
building_risk.select(
    "building_id",
    "current_address",
    "building_risk_score"
).orderBy(
    F.desc("building_risk_score")
).show(
    20,
    truncate=False
)

+-----------+----------------------------+-------------------+
|building_id|current_address             |building_risk_score|
+-----------+----------------------------+-------------------+
|BLD:1053262|2 WEST 120 STREET           |97.88              |
|BLD:1060413|2400 ADAM C POWELL BOULEVARD|97.49              |
|BLD:1061246|1661 AMSTERDAM AVENUE       |97.45              |
|BLD:1062311|3427 BROADWAY               |97.44              |
|BLD:1063255|600 WEST 157 STREET         |97.31              |
|BLD:1082387|326 EAST 100 STREET         |97.25              |
|BLD:3077973|1302 JEFFERSON AVENUE       |97.25              |
|BLD:2016413|382 EAST 199 STREET         |97.19              |
|BLD:4006216|23-35 29 AVENUE             |97.06              |
|BLD:1060908|105 EDGECOMBE AVENUE        |97.0               |
|BLD:2013582|2381 VALENTINE AVENUE       |96.94              |
|BLD:2003376|1360 OGDEN AVENUE           |96.75              |
|BLD:1058264|181 WEST 135 STREET         |96.69        

In [22]:
# ==================================================
# FINAL PROPERTY RISK SCORE
# ==================================================

PROPERTY_ONLY_UPLIFT_FACTOR = 0.50


property_risk_scored = (
    property_risk_normalized

    .withColumn(
        "property_risk_score",

        # Case 1:
        # Building Risk + Property-only Risk
        F.when(
            F.col("max_building_risk_score").isNotNull()
            & F.col("property_only_risk_score").isNotNull(),

            F.col("max_building_risk_score")
            +
            (
                F.lit(100.0)
                - F.col("max_building_risk_score")
            )
            *
            (
                F.col("property_only_risk_score")
                / F.lit(100.0)
            )
            *
            F.lit(PROPERTY_ONLY_UPLIFT_FACTOR)
        )

        # Case 2:
        # Building Risk only
        .when(
            F.col("max_building_risk_score").isNotNull(),
            F.col("max_building_risk_score")
        )

        # Case 3:
        # Property-only Risk only
        .when(
            F.col("property_only_risk_score").isNotNull(),
            F.col("property_only_risk_score")
        )
    )

    .withColumn(
        "property_risk_score",
        F.round(
            F.least(
                F.col("property_risk_score"),
                F.lit(100.0)
            ),
            2
        )
    )
)

In [23]:
property_risk_scored = (
    property_risk_scored

    .withColumn(
        "property_risk_source",

        F.when(
            F.col("max_building_risk_score").isNotNull()
            & F.col("property_only_risk_score").isNotNull(),
            F.lit("BUILDING_PLUS_PROPERTY_EVENTS")
        )

        .when(
            F.col("max_building_risk_score").isNotNull(),
            F.lit("BUILDING_ONLY")
        )

        .when(
            F.col("property_only_risk_score").isNotNull(),
            F.lit("PROPERTY_EVENTS_ONLY")
        )

        .otherwise(
            F.lit("NO_RISK_DATA")
        )
    )
)

In [24]:
print(
    "Properties with final Risk:",
    property_risk_scored
    .filter(
        F.col("property_risk_score").isNotNull()
    )
    .count()
)

print(
    "Properties without Risk:",
    property_risk_scored
    .filter(
        F.col("property_risk_score").isNull()
    )
    .count()
)

(
    property_risk_scored
    .groupBy("property_risk_source")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

Properties with final Risk: 858284


Properties without Risk: 0


+-----------------------------+------+
|property_risk_source         |count |
+-----------------------------+------+
|NO_RISK_DATA                 |673369|
|BUILDING_ONLY                |171050|
|PROPERTY_EVENTS_ONLY         |13333 |
|BUILDING_PLUS_PROPERTY_EVENTS|532   |
+-----------------------------+------+



In [25]:
property_final_quantiles = (
    property_risk_scored
    .filter(
        F.col("property_risk_score").isNotNull()
    )
    .approxQuantile(
        "property_risk_score",
        [0.50, 0.75, 0.90, 0.95, 0.99],
        0.001
    )
)

print("Final Property Risk Distribution")
print("P50 =", property_final_quantiles[0])
print("P75 =", property_final_quantiles[1])
print("P90 =", property_final_quantiles[2])
print("P95 =", property_final_quantiles[3])
print("P99 =", property_final_quantiles[4])

Final Property Risk Distribution
P50 = 100.0
P75 = 100.0
P90 = 100.0
P95 = 100.0
P99 = 100.0


In [26]:
# ==================================================
# FINAL PROPERTY RISK SCORE
# ==================================================

PROPERTY_ONLY_UPLIFT_FACTOR = 0.50


property_risk_scored = (
    property_risk_normalized

    .withColumn(
        "property_risk_score_raw",

        # Building Risk + Property-only Risk
        F.when(
            F.col("max_building_risk_score").isNotNull()
            & F.col("property_only_risk_score").isNotNull(),

            F.col("max_building_risk_score")
            +
            (
                F.lit(100.0)
                - F.col("max_building_risk_score")
            )
            *
            (
                F.col("property_only_risk_score")
                / F.lit(100.0)
            )
            *
            F.lit(PROPERTY_ONLY_UPLIFT_FACTOR)
        )

        # Building Risk only
        .when(
            F.col("max_building_risk_score").isNotNull(),
            F.col("max_building_risk_score")
        )

        # Property-only Risk only
        .when(
            F.col("property_only_risk_score").isNotNull(),
            F.col("property_only_risk_score")
        )

        # No data
        .otherwise(F.lit(None).cast("double"))
    )

    .withColumn(
        "property_risk_score",

        F.when(
            F.col("property_risk_score_raw").isNotNull(),

            F.round(
                F.least(
                    F.col("property_risk_score_raw"),
                    F.lit(100.0)
                ),
                2
            )
        )
        .otherwise(F.lit(None).cast("double"))
    )
)

In [27]:
property_risk_scored = (
    property_risk_scored

    .withColumn(
        "property_risk_source",

        F.when(
            F.col("max_building_risk_score").isNotNull()
            & F.col("property_only_risk_score").isNotNull(),
            F.lit("BUILDING_PLUS_PROPERTY_EVENTS")
        )

        .when(
            F.col("max_building_risk_score").isNotNull(),
            F.lit("BUILDING_ONLY")
        )

        .when(
            F.col("property_only_risk_score").isNotNull(),
            F.lit("PROPERTY_EVENTS_ONLY")
        )

        .otherwise(
            F.lit("NO_RISK_DATA")
        )
    )
)

In [28]:
print(
    "Properties with final Risk:",
    property_risk_scored
    .filter(F.col("property_risk_score").isNotNull())
    .count()
)

print(
    "Properties without Risk:",
    property_risk_scored
    .filter(F.col("property_risk_score").isNull())
    .count()
)

(
    property_risk_scored
    .groupBy("property_risk_source")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

Properties with final Risk: 184915


Properties without Risk: 673369


+-----------------------------+------+
|property_risk_source         |count |
+-----------------------------+------+
|NO_RISK_DATA                 |673369|
|BUILDING_ONLY                |171050|
|PROPERTY_EVENTS_ONLY         |13333 |
|BUILDING_PLUS_PROPERTY_EVENTS|532   |
+-----------------------------+------+



In [29]:
property_final_quantiles = (
    property_risk_scored
    .filter(
        F.col("property_risk_score").isNotNull()
    )
    .approxQuantile(
        "property_risk_score",
        [0.50, 0.75, 0.90, 0.95, 0.99],
        0.001
    )
)

print("Final Property Risk Distribution")
print("P50 =", property_final_quantiles[0])
print("P75 =", property_final_quantiles[1])
print("P90 =", property_final_quantiles[2])
print("P95 =", property_final_quantiles[3])
print("P99 =", property_final_quantiles[4])

Final Property Risk Distribution
P50 = 16.21
P75 = 24.82
P90 = 45.45
P95 = 59.67
P99 = 78.96


In [30]:
# ==================================================
# PROPERTY RISK LEVEL
# ==================================================

PROPERTY_P75 = 24.82
PROPERTY_P95 = 59.67
PROPERTY_P99 = 78.96


property_risk_final = (
    property_risk_scored

    .withColumn(
        "property_risk_level",

        F.when(
            F.col("property_risk_score").isNull(),
            F.lit("NO_DATA")
        )

        .when(
            F.col("property_risk_score") < PROPERTY_P75,
            F.lit("LOW")
        )

        .when(
            F.col("property_risk_score") < PROPERTY_P95,
            F.lit("MEDIUM")
        )

        .when(
            F.col("property_risk_score") < PROPERTY_P99,
            F.lit("HIGH")
        )

        .otherwise(
            F.lit("CRITICAL")
        )
    )

    .withColumn(
        "risk_as_of_date",
        F.lit(str(risk_as_of_date)).cast("date")
    )
)

In [31]:
# ==================================================
# VALIDATE PROPERTY RISK LEVELS
# ==================================================

(
    property_risk_final
    .groupBy("property_risk_level")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+-------------------+------+
|property_risk_level|count |
+-------------------+------+
|NO_DATA            |673369|
|LOW                |138594|
|MEDIUM             |36984 |
|HIGH               |7352  |
|CRITICAL           |1985  |
+-------------------+------+



In [32]:
# ==================================================
# FINAL PROPERTY RISK DATASET
# Grain: 1 row = 1 property_id
# ==================================================

property_risk_output = (
    property_risk_final

    .select(
        # Identity
        "property_id",
        "bbl",
        "property_address",
        "borough",
        "zipcode",
        "latitude",
        "longitude",

        # Final Risk
        "property_risk_score",
        "property_risk_level",
        "property_risk_source",
        "risk_as_of_date",

        # Building context
        "building_count",
        "max_building_risk_score",
        "avg_building_risk_score",

        # Property-only score
        "property_only_risk_score",

        # Normalized components
        "property_score_311",
        "property_score_hpd_severity",
        "property_score_hpd_recency",

        # Raw signals
        "property_risk_311_raw",
        "property_risk_hpd_severity_raw",
        "property_risk_hpd_recency_raw",

        # 311 features
        "property_311_0_30",
        "property_311_31_90",
        "property_311_91_365",

        # HPD features
        "property_hpd_open_class_a",
        "property_hpd_open_class_b",
        "property_hpd_open_class_c",
        "property_hpd_0_30",
        "property_hpd_31_90",
        "property_hpd_91_365",

        # Property attributes
        "yearbuilt",
        "numbldgs",
        "numfloors",
        "unitsres",
        "unitstotal",
        "landuse",
        "bldgclass"
    )
)

In [33]:
print(
    "Final rows:",
    property_risk_output.count()
)

print(
    "Distinct property_id:",
    property_risk_output
    .select("property_id")
    .distinct()
    .count()
)

print(
    "With Risk:",
    property_risk_output
    .filter(F.col("property_risk_score").isNotNull())
    .count()
)

print(
    "Without Risk:",
    property_risk_output
    .filter(F.col("property_risk_score").isNull())
    .count()
)

print(
    "Risk > 100:",
    property_risk_output
    .filter(F.col("property_risk_score") > 100)
    .count()
)

print(
    "Risk < 0:",
    property_risk_output
    .filter(F.col("property_risk_score") < 0)
    .count()
)

Final rows: 858284
Distinct property_id: 858284


With Risk: 184915


Without Risk: 673369


Risk > 100: 0


Risk < 0: 0


In [34]:
# ==================================================
# SAVE FINAL PROPERTY RISK
# ==================================================

PROPERTY_RISK_PATH = minio_path(
    "gold/property_risk/property_risk_score"
)

(
    property_risk_output
    .write
    .mode("overwrite")
    .parquet(PROPERTY_RISK_PATH)
)

print("Saved Property Risk to:", PROPERTY_RISK_PATH)

26/09/12 14:40:36 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Saved Property Risk to: s3a://nyc-building-risk/gold/property_risk/property_risk_score


In [35]:
# ==================================================
# READ-BACK VALIDATION
# ==================================================

property_risk_saved = spark.read.parquet(
    PROPERTY_RISK_PATH
)

print(
    "Saved rows:",
    property_risk_saved.count()
)

print(
    "Saved distinct property_id:",
    property_risk_saved
    .select("property_id")
    .distinct()
    .count()
)

property_risk_saved.groupBy(
    "property_risk_level"
).count().orderBy(
    F.desc("count")
).show()

Saved rows: 858284
Saved distinct property_id: 858284
+-------------------+------+
|property_risk_level| count|
+-------------------+------+
|            NO_DATA|673369|
|                LOW|138594|
|             MEDIUM| 36984|
|               HIGH|  7352|
|           CRITICAL|  1985|
+-------------------+------+

